In [3]:
# scripts/03_make_weights_custom.py
from pathlib import Path
import pandas as pd, numpy as np

rets = pd.read_parquet("data/processed/returns_linear.parquet")
assets = [c for c in rets.columns if c!="SPY"]
meta = pd.read_csv("data/raw/meta_tickers.csv").set_index("ticker")["sector"].to_dict()

# target sector tilts (ví dụ cố ý khác SPY)
target = {
    "XLK": 0.40,  # Tech overweight
    "XLY": 0.20,
    "XLC": 0.10,
    "XLF": 0.05,
    "XLV": 0.10,
    "XLE": 0.03,
    "XLI": 0.05,
    "XLP": 0.03,
    "XLU": 0.04
}
# normalize
s = sum(target.values()); target = {k:v/s for k,v in target.items()}

# tháng-end index
idxM = rets.resample("ME").last().index
rows=[]
for ts in idxM:
    w = pd.Series(0.0, index=assets)
    for sct, tgt in target.items():
        tick = [t for t in assets if meta.get(t)==sct]
        if tick:
            w.loc[tick] = tgt/len(tick)
    w.name = ts
    rows.append(w)

w_monthly = pd.DataFrame(rows).sort_index()
Path("data/portfolio_weights").mkdir(parents=True, exist_ok=True)
w_monthly.to_parquet("data/portfolio_weights/custom_monthly_wide.parquet")